# OLTP Transformation Testing

### Validate the incremental OLTP SQL before the transformation is moved into the production SQL runner and Airflow.

In [1]:
# Import sys so Python can locate project modules.
import sys

# Import Path for safe filesystem handling.
from pathlib import Path


# The notebook lives inside /notebooks, so the project root is one level above.
project_root = Path.cwd().parent


# Add the project root to Python's module search path if necessary.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Display the detected project root.
print("Project root:", project_root)

Project root: /Users/mac/Documents/netflix-data-engineering


## Import database connection

In [2]:
# Import the shared production PostgreSQL connection helper.
from src.database import get_etl_connection

## Test 1: Locate the staged titles batch

Find the latest real titles batch currently available in staging.

In [3]:
# Open the ETL database connection.
with get_etl_connection() as connection:

    # Open a cursor for the batch lookup.
    with connection.cursor() as cursor:

        # Find the latest batch that loaded titles.csv.
        cursor.execute(
            """
            SELECT
                batch_id,
                file_name,
                rows_received,
                status
            FROM etl.batch_history
            WHERE file_name = 'titles.csv'
            ORDER BY batch_id DESC
            LIMIT 1;
            """
        )

        # Retrieve the latest matching batch.
        latest_titles_batch = cursor.fetchone()


# Stop if no titles batch exists.
assert latest_titles_batch is not None


# Extract the returned values.
titles_batch_id = latest_titles_batch[0]
titles_file_name = latest_titles_batch[1]
titles_rows_received = latest_titles_batch[2]
titles_batch_status = latest_titles_batch[3]


# Display the batch information.
print("Batch ID:", titles_batch_id)
print("File:", titles_file_name)
print("Rows received:", titles_rows_received)
print("Current status:", titles_batch_status)

Batch ID: 12
File: titles.csv
Rows received: 5850
Current status: RUNNING


## Test 2: Capture the OLTP baseline

Record the current number of rows in the title table before running the incremental upsert.

In [5]:
# Open a fresh PostgreSQL connection.
with get_etl_connection() as connection:

    # Open a cursor for the baseline count.
    with connection.cursor() as cursor:

        # Count the current OLTP title records.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title;
            """
        )

        # Store the baseline count.
        title_count_before = cursor.fetchone()[0]


# Display the baseline.
print("Title rows before upsert:", title_count_before)

Title rows before upsert: 5849


## Test 3: Execute the incremental title upsert

Run the production OLTP title transformation for the staged titles batch using the real batch ID.

In [6]:
# Build the path to the production title upsert SQL file.
upsert_titles_sql_path = (
    project_root
    / "sql"
    / "02_oltp"
    / "01_upsert_titles.sql"
)


# Confirm that the production SQL file exists before trying to run it.
assert upsert_titles_sql_path.exists(), (
    f"SQL file not found: {upsert_titles_sql_path}"
)


# Read the SQL script into memory.
upsert_titles_sql = upsert_titles_sql_path.read_text(
    encoding="utf-8"
)


# Open the ETL database connection.
with get_etl_connection() as connection:

    # Open a cursor for executing the transformation.
    with connection.cursor() as cursor:

        # Execute the SQL and supply the real batch ID
        # to the %(batch_id)s placeholder inside the SQL file.
        cursor.execute(
            upsert_titles_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    # Commit the complete OLTP transformation.
    connection.commit()


# Confirm successful execution.
print(
    f"PASS: batch {titles_batch_id} transformed into the OLTP title table."
)

PASS: batch 12 transformed into the OLTP title table.


## Test 4: Verify the OLTP result

Compare the title table before and after the incremental transformation.

In [7]:
# Open a fresh database connection for independent verification.
with get_etl_connection() as connection:

    # Open a cursor for the validation query.
    with connection.cursor() as cursor:

        # Count the OLTP title records after the upsert.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title;
            """
        )

        # Store the resulting count.
        title_count_after = cursor.fetchone()[0]


# Display the before/after comparison.
print("Title rows before:", title_count_before)
print("Title rows after: ", title_count_after)

Title rows before: 5849
Title rows after:  5849


## Test 5: Validate staged title coverage

Confirm that every valid title ID from the current staging batch exists in the OLTP title table.

In [8]:
# Open a fresh connection for the coverage check.
with get_etl_connection() as connection:

    # Open a cursor for validation.
    with connection.cursor() as cursor:

        # Count staged title IDs from this batch that are missing in OLTP.
        cursor.execute(
            """
            SELECT COUNT(*)

            FROM staging.titles_raw AS staging

            WHERE staging.batch_id = %s

              AND staging.id IS NOT NULL

              AND staging.id <> ''

              AND staging.title IS NOT NULL

              AND staging.title <> ''

              AND NOT EXISTS (
                  SELECT 1
                  FROM title AS oltp
                  WHERE oltp.title_id = staging.id
              );
            """,
            (titles_batch_id,),
        )

        # Retrieve the number of missing titles.
        missing_titles_count = cursor.fetchone()[0]


# Display the quality-check result.
print(
    "Valid staged titles missing from OLTP:",
    missing_titles_count,
)


# No valid staged title should be absent after the upsert.
assert missing_titles_count == 0


# Confirm successful coverage.
print(
    "PASS: all valid staged titles are present in OLTP."
)

Valid staged titles missing from OLTP: 0
PASS: all valid staged titles are present in OLTP.


## Test 6: Idempotency test

Re-run the same batch and confirm that the title table does not gain duplicate rows.

In [9]:
# Execute the exact same production SQL again.
with get_etl_connection() as connection:

    # Open a cursor for the rerun.
    with connection.cursor() as cursor:

        # Reprocess the same staged batch.
        cursor.execute(
            upsert_titles_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    # Commit the rerun.
    connection.commit()


# Count the OLTP titles after the second execution.
with get_etl_connection() as connection:

    # Open a validation cursor.
    with connection.cursor() as cursor:

        # Count the title table.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title;
            """
        )

        # Store the count after rerunning the same batch.
        title_count_after_rerun = cursor.fetchone()[0]


# Display both post-transformation counts.
print("After first run:", title_count_after)
print("After rerun:    ", title_count_after_rerun)


# Reprocessing the same batch must not create more title records.
assert title_count_after_rerun == title_count_after


# Confirm idempotent behaviour.
print(
    "PASS: title upsert is idempotent."
)

After first run: 5849
After rerun:     5849
PASS: title upsert is idempotent.
